# Reed-Solomon Decoder

In [64]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
gf_extended = GaloisField(4, 0b10011)
root        = [GFE1, 1, 2, 3, 4, 5]

def do_unpack(value, degree, gf):
    # useful return of a x**degree poly
    if value == -1:
        ak  = [GFE0]*degree + [GFE1]
    else:
        mask    = gf._order
        ak      = [(value >> i*gf._field_m) & mask for i in range(degree+1)]
        ak      = [gf._get_power(a) for a in ak]
    return GaloisPoly(gf, ak[::-1])

def get_discrepancy(mu, syn, Lx):
    d_mu    = syn[mu]
    for i in range(1, Lx._degree+1):
        term    = gf_extended.mul(Lx @ i, syn[mu - i])[ID_POWER]
        d_mu    = gf_extended.add(d_mu, term)[ID_POWER]
    return d_mu

def get_locator_poly(syn):
    Lx      = []
    mu_lmu  = []
    dmu     = []

    # initialization, locator = 1
    poly = do_unpack(-1, 0, gf_extended)
    Lx.append(poly)

    for mu in range(2*t):
        mu_lmu.append(mu - Lx[-1]._degree)

        disc = get_discrepancy(mu, syn, Lx[-1])
        # if dicrepancy zero, locator adjust not needed
        if disc == GFE0:
            poly = Lx[-1]
        else:
            if mu == 0:
                # first adjustment: L1 = 1 + d0*x always (because dm1 = 1)
                xm      = do_unpack(-1, 1, gf_extended)
                poly    = Lx[-1] + (xm * disc)
            else:
                # to adjust we use all previous rho, except newly calculated
                rho     = mu_lmu.index(max(mu_lmu[:-1]))
                xm      = do_unpack(-1, mu-rho, gf_extended)
                xm      = xm * gf_extended.div(disc, dmu[rho])[ID_POWER]
                poly    = Lx[-1] + (xm * Lx[rho])

        dmu.append(disc)
        Lx.append(poly)
        print(Lx[-1], "\n")

    return Lx[-1]

Field Closed Succesfully!, 15 Non-Zero Elements


## Get Syndromes

In [ ]:
# r   = 0x52D0A_F17CB_1EE64
r   = 0x56D09_F171B_1E06F
r   = do_unpack(r, 14, gf_extended)

syn = [r(a) for a in root]

syn


[13, 2, 7, 10, 9, 5]

## Berlekamp-Massey Algorithm

In [114]:
t   = 3
Lx  = get_locator_poly(syn)


a13*x + 1 

a4*x + 1 

a12*x2 + a4*x + 1 

a12*x2 + a4*x + 1 

a12*x2 + a4*x + 1 

a12*x2 + a4*x + 1 



## Chien Algorithm (wip)

In [ ]:
# loc_root    = [13, 5]
loc_root    = [GFE1, 2, 9]

## Forney Algorithm

In [116]:
Sx  = GaloisPoly(gf_extended, syn[::-1])
Zx  = Sx * Lx
Zx  = Zx % do_unpack(-1, Lx._degree, gf_extended)
Lxp = Lx.formal_derivative()

print(Sx, Zx, Lxp)

lo  = 0
for rk in loc_root:
    num = gf_extended.mul(Zx(rk), gf_extended.pow(rk, lo-1)[ID_POWER])[ID_POWER]
    den = Lxp(rk)
    ek  = gf_extended.div(num, den)[ID_POWER]
    print(ek)

a5*x5 + a9*x4 + a10*x3 + a7*x2 + a2*x + a13
 a13
 a4

11
4


In [75]:
print(Sx % do_unpack(0x1000, 3, gf_extended))

x2 + a14*x + a1



In [107]:
gf_extended.pow(2, -2)

(14, 11)